In [29]:
import os
import cv2
import numpy as np
import pandas as pd

from skimage.feature import hog, local_binary_pattern

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [30]:
def extract_hog(image):

    features = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    return features

In [31]:
def extract_lbp(image):

    radius = 1
    points = 8

    lbp = local_binary_pattern(
        image,
        points,
        radius,
        method="uniform"
    )

    # Histogram
    histogram, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, points + 3),
        range=(0, points + 2)
    )

    # Normalize
    histogram = histogram.astype("float")
    histogram /= (histogram.sum() + 1e-7)

    return histogram

In [32]:
def extract_features(image_path):

    image = cv2.imread(image_path)

    if image is None:
        return None

    # Resize
    image = cv2.resize(image, (128, 128))

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # HOG
    hog_features = extract_hog(gray)

    # LBP
    lbp_features = extract_lbp(gray)

    # Combine
    features = np.concatenate([
        hog_features,
        lbp_features
    ])

    return features

In [33]:
def create_dataset(split, max_images_per_class=20):

    data = []

    split_path = os.path.join('/mnt/d/ImageProcessing_Dataset/deepfake/realvsfake', split)

    classes = {
        "real": 0,
        "fake": 1
    }

    for class_name, label in classes.items():

        class_path = os.path.join(split_path, class_name)

        files = os.listdir(class_path)

        # Only image files
        files = [
            f for f in files
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]

        # Take only the number we want
        files = files[:max_images_per_class]

        print(
            split,
            class_name,
            "images:",
            len(files)
        )

        for filename in files:

            image_path = os.path.join(
                class_path,
                filename
            )

            features = extract_features(image_path)

            if features is not None:

                row = list(features)
                row.append(label)

                data.append(row)

    return np.array(data)

In [34]:
train_data = create_dataset(
    "train",
    max_images_per_class=20
)

train real images: 20
train fake images: 20


In [35]:
val_data = create_dataset(
    "val",
    max_images_per_class=5
)

test_data = create_dataset(
    "test",
    max_images_per_class=5
)

val real images: 5
val fake images: 5
test real images: 5
test fake images: 5


In [36]:
feature_count = train_data.shape[1] - 1

columns = [
    f"feature_{i}"
    for i in range(feature_count)
]

columns.append("label")

In [37]:
train_df = pd.DataFrame(
    train_data,
    columns=columns
)

val_df = pd.DataFrame(
    val_data,
    columns=columns
)

test_df = pd.DataFrame(
    test_data,
    columns=columns
)

In [38]:
train_df.to_csv(
    "train_features.csv",
    index=False
)

val_df.to_csv(
    "val_features.csv",
    index=False
)

test_df.to_csv(
    "test_features.csv",
    index=False
)

In [39]:
X_train = train_df.drop("label", axis=1)
y_train = train_df["label"]

X_val = val_df.drop("label", axis=1)
y_val = val_df["label"]

X_test = test_df.drop("label", axis=1)
y_test = test_df["label"]

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

In [41]:
model = SVC(
    kernel="rbf",
    C=10,
    probability=True
)

model.fit(
    X_train_scaled,
    y_train
)

,C,10
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [42]:
val_predictions = model.predict(X_val_scaled)

print("Validation Accuracy:",
      accuracy_score(y_val, val_predictions))

print(
    classification_report(
        y_val,
        val_predictions,
        target_names=["Real", "Fake"]
    )
)

Validation Accuracy: 0.8
              precision    recall  f1-score   support

        Real       0.71      1.00      0.83         5
        Fake       1.00      0.60      0.75         5

    accuracy                           0.80        10
   macro avg       0.86      0.80      0.79        10
weighted avg       0.86      0.80      0.79        10



In [43]:
test_predictions = model.predict(X_test_scaled)

print("Test Accuracy:",
      accuracy_score(y_test, test_predictions))

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=["Real", "Fake"]
    )
)

Test Accuracy: 0.6
              precision    recall  f1-score   support

        Real       0.60      0.60      0.60         5
        Fake       0.60      0.60      0.60         5

    accuracy                           0.60        10
   macro avg       0.60      0.60      0.60        10
weighted avg       0.60      0.60      0.60        10



In [44]:
print("Training labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Training labels:
label
0.0    20
1.0    20
Name: count, dtype: int64

Validation labels:
label
0.0    5
1.0    5
Name: count, dtype: int64

Test labels:
label
0.0    5
1.0    5
Name: count, dtype: int64


In [45]:
print("Actual:")
print(y_val.values)

print("\nPredicted:")
print(val_predictions)

Actual:
[0. 0. 0. 0. 0. 1. 1. 1. 1. 1.]

Predicted:
[0. 0. 0. 0. 0. 1. 1. 1. 0. 0.]


In [46]:
train_predictions = model.predict(X_train_scaled)

print(
    "Training Accuracy:",
    accuracy_score(y_train, train_predictions)
)

Training Accuracy: 1.0


In [47]:
print("Number of features:", X_train.shape[1])

print("First image feature values:")
print(X_train.iloc[0].values[:20])

print("\nSecond image feature values:")
print(X_train.iloc[1].values[:20])

Number of features: 8110
First image feature values:
[0.22617751 0.16811105 0.15801888 0.11145129 0.10617941 0.06589574
 0.00715049 0.05214802 0.03683608 0.15838203 0.27353353 0.24117183
 0.2231649  0.16207723 0.03391777 0.01430099 0.01130592 0.07374843
 0.16114614 0.22122595]

Second image feature values:
[0.09324533 0.         0.07130641 0.05332348 0.14938604 0.18041883
 0.03956064 0.02507373 0.00982892 0.29382858 0.06975574 0.15261401
 0.05440832 0.07770444 0.04652787 0.01758251 0.03205698 0.1504791
 0.         0.01120669]


In [48]:
print("Feature statistics:")
print(X_train.describe())

Feature statistics:


       feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
count  40.000000  40.000000  40.000000  40.000000  40.000000  40.000000   
mean    0.198613   0.082005   0.113437   0.090921   0.204891   0.054131   
std     0.095663   0.090837   0.097449   0.101480   0.112710   0.052167   
min     0.031658   0.000000   0.000000   0.000000   0.014347   0.000000   
25%     0.109191   0.007069   0.025210   0.025407   0.104957   0.017044   
50%     0.221348   0.046530   0.091221   0.067704   0.229808   0.036762   
75%     0.271737   0.109734   0.173581   0.105491   0.259457   0.080123   
max     0.381819   0.286616   0.320995   0.485951   0.485951   0.196311   

       feature_6  feature_7  feature_8  feature_9  ...  feature_8100  \
count  40.000000  40.000000  40.000000  40.000000  ...     40.000000   
mean    0.092462   0.087700   0.081806   0.221998  ...      0.043793   
std     0.091910   0.098158   0.121710   0.095125  ...      0.009995   
min     0.000000   0.000000   0.0000

In [49]:
real_features = X_train[y_train == 0]
fake_features = X_train[y_train == 1]

print("Real feature mean:")
print(real_features.mean().mean())

print("\nFake feature mean:")
print(fake_features.mean().mean())

Real feature mean:
0.1209440293837626

Fake feature mean:
0.12345267856449076


In [50]:
model_test = SVC(
    kernel="rbf",
    C=10
)

model_test.fit(X_train, y_train)

train_pred = model_test.predict(X_train)
val_pred = model_test.predict(X_val)

print("Training Accuracy:",
      accuracy_score(y_train, train_pred))

print("Validation Accuracy:",
      accuracy_score(y_val, val_pred))

Training Accuracy: 1.0
Validation Accuracy: 0.8
